In [91]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [92]:
df = pd.read_csv('qoute_dataset.csv')
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [93]:
df.shape

(3038, 2)

In [94]:
quotes=df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [95]:
quotes = quotes.str.lower()

In [96]:
import string
translator = str.maketrans('','',string.punctuation)
quotes= quotes.apply(lambda x: x.translate(translator))
quotes.head()


0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: object

In [97]:
# tokenizing and it make vocabulart using number to every word
from tensorflow.keras.preprocessing.text import Tokenizer

In [98]:
vocab_size = 8978
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)


In [99]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [100]:
sequence = tokenizer.texts_to_sequences(quotes)

In [101]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [102]:
for i in range(3):
    print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [ ]:
# input and output varible(we have to predict next word)
x = []
y = []
for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        x.append(input_seq)
        y.append(output_seq)
print(x
     
     )


In [104]:
# now we have to fix the size use padding so every sentence should be same length

In [105]:
max_len = max(len(x) for x in x)
max_len

745

In [106]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_padded = pad_sequences(x, maxlen=max_len, padding='pre')  
x_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [107]:
y = np.array(y)
y.shape

(85270,)

In [108]:
# one hot encoding
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes = vocab_size)
y.shape
y_one_hot.shape

(85270, 8978)

In [109]:
# embedings using rnn
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM , Dense

In [110]:
embeding_dim = 50 
rnn_units = 128

In [111]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embeding_dim,
             input_length = max_len)
)
rnn_model.add(SimpleRNN(units= rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

In [112]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [113]:
rnn_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [114]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embeding_dim,input_length=max_len)
)
lstm_model.add(LSTM(units= rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [115]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [116]:
lstm_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [117]:
epochs = 10
batch_size = 128

In [118]:
history_rnn = rnn_model.fit(
    x_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

In [ ]:
epochs = 100
batch_size = 128
history_lstm = lstm_model.fit(
    x_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1

)

Epoch 1/100
 81/600 ━━━━━━━━━━━━━━━━━━━━ 9:09 1s/step - accuracy: 0.0251 - loss: 8.1694